In [1]:
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer
from pathlib import Path

2026-07-08 23:00:02.406160276 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
path="models/Xenova/all-MiniLM-L6-v2"
path = Path(path)
path

PosixPath('models/Xenova/all-MiniLM-L6-v2')

In [3]:
tokenizer = Tokenizer.from_file(str(path / "tokenizer.json"))
tokenizer

Tokenizer(version="1.0", truncation=TruncationParams(direction=Right, max_length=128, strategy=LongestFirst, stride=0), padding=PaddingParams(strategy=Fixed(128), direction=Right, pad_to_multiple_of=None, pad_id=0, pad_type_id=0, pad_token="[PAD]"), added_tokens=[{"id":0, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":100, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":101, "content":"[CLS]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":102, "content":"[SEP]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":103, "content":"[MASK]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=True), pre_tokenizer=BertPreTokenize

In [4]:
session = ort.InferenceSession(
            str(path / "model.onnx"), providers=["CPUExecutionProvider"]
        )
session

In [5]:
input_names = {inp.name for inp in session.get_inputs()}
input_names

{'attention_mask', 'input_ids', 'token_type_ids'}

In [6]:
tokenizer.enable_padding()

In [7]:
texts = ["Hello world", "This is a test", "!"]

In [8]:
encoded = tokenizer.encode_batch(texts)
encoded

[Encoding(num_tokens=6, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=6, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=6, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [9]:
feed = {}
if "input_ids" in input_names:
    feed["input_ids"] = np.array([e.ids for e in encoded], dtype=np.int64)
if "attention_mask" in input_names:
    feed["attention_mask"] = np.array(
        [e.attention_mask for e in encoded], dtype=np.int64
    )
if "token_type_ids" in input_names:
    feed["token_type_ids"] = np.array(
        [e.type_ids for e in encoded], dtype=np.int64
    )

In [10]:
feed

{'input_ids': array([[ 101, 7592, 2088,  102,    0,    0],
        [ 101, 2023, 2003, 1037, 3231,  102],
        [ 101,  999,  102,    0,    0,    0]]),
 'attention_mask': array([[1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0]]),
 'token_type_ids': array([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]])}

In [20]:
len(session.run(None, feed))

1

In [19]:
hidden = session.run(None, feed)[0]
hidden.shape

(3, 6, 384)

In [21]:
feed["attention_mask"]

array([[1, 1, 1, 1, 0, 0],
       [1, 1, 1, 1, 1, 1],
       [1, 1, 1, 0, 0, 0]])

In [22]:
feed["attention_mask"][..., None]

array([[[1],
        [1],
        [1],
        [1],
        [0],
        [0]],

       [[1],
        [1],
        [1],
        [1],
        [1],
        [1]],

       [[1],
        [1],
        [1],
        [0],
        [0],
        [0]]])

In [23]:
mask = feed["attention_mask"][..., None]
mask.shape

(3, 6, 1)

In [24]:
mask.sum(axis=1)

array([[4],
       [6],
       [3]])

In [26]:
(hidden * mask).sum(axis=1).shape

(3, 384)

In [27]:
pooled = (hidden * mask).sum(axis=1) / mask.sum(axis=1)
pooled.shape

(3, 384)

In [29]:
np.linalg.norm(pooled, axis=1, keepdims=True)


array([[5.72684917],
       [5.4449748 ],
       [5.85597154]])

In [30]:
pooled = pooled / np.linalg.norm(pooled, axis=1, keepdims=True)
pooled

array([[-0.03447729,  0.03102317,  0.00673498, ..., -0.0228898 ,
         0.03893759,  0.03020687],
       [ 0.03061244,  0.01383139, -0.0208438 , ..., -0.01682046,
         0.09417654, -0.04484286],
       [-0.1350287 , -0.01453779,  0.01028815, ...,  0.04820148,
        -0.05002563,  0.02239895]], shape=(3, 384))

In [31]:
pooled.shape

(3, 384)

In [37]:
a = np.arange(6).reshape(2, 3)
a.shape

(2, 3)

In [38]:
a[None, :, :].shape

(1, 2, 3)

In [39]:
a[None, ...].shape

(1, 2, 3)

In [40]:
a[:, None, :].shape

(2, 1, 3)

In [41]:
a[:, None, :]

array([[[0, 1, 2]],

       [[3, 4, 5]]])

In [42]:
a[..., None]

array([[[0],
        [1],
        [2]],

       [[3],
        [4],
        [5]]])